# 04 — Model evaluation and annual scoring contract

This notebook explains and checks the completed nine-feature continuous-model evaluation. The retained hurdle is a continuous comparative research artifact. It loads the recorded train/validation/final-test evidence and the reusable operational artefact; it does **not** tune, select, or retrain a model.

The output is an estimated next-year burned share, not a probability, safety score, or purchase recommendation.

In [ ]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook_support import read_json_artifact, require_artifacts, resolve_project_root
PROJECT_ROOT = resolve_project_root(PROJECT_ROOT)
from src.modeling import NINE_FEATURES
from src.feature_contract import TARGET_COLUMN

metrics = read_json_artifact(PROJECT_ROOT, 'data/processed/extended_model_selection_2010_2021/final_temporal_test_metrics.json')
model_path = require_artifacts(PROJECT_ROOT, ['data/processed/final_model_2010_2024/nine_feature_hurdle.joblib'])[0]
print('Loaded frozen evaluation evidence and current operational model artifact.')

## Fixed data split and feature contract

Candidate selection used development and validation periods before one frozen final temporal test. This notebook reports that result; it cannot use the final-test years to change the selected method.

In [ ]:
payload = joblib.load(model_path)
assert metrics['design']['final_test_years'] == [2022, 2023, 2024]
assert metrics['design']['tuning_performed'] is False
assert payload['feature_order'] == list(NINE_FEATURES)

contract = pd.DataFrame({'feature': NINE_FEATURES, 'role': ['predictor'] * len(NINE_FEATURES)})
display(contract)
print('Target:', TARGET_COLUMN)
print('Operational training predictor years:', payload['training_predictor_years'])
print('Final test accessed only for fixed evaluation:', metrics['design']['final_test_years'])

## Held-out comparison

MAE and RMSE measure error over all rows; positive-row MAE focuses on rows where fire occurred. Capture at 20% is a technical ranking diagnostic, not a buyer threshold.

In [ ]:
rows = []
for model_name, result in metrics['metrics'].items():
    overall = result['overall']
    ranking = metrics['tie_aware_ranking_diagnostics'][model_name]['overall']['top_20_percent']
    rows.append({
        'model': model_name,
        'MAE': overall['mae_all'],
        'RMSE': overall['rmse_all'],
        'positive_row_MAE': overall['mae_positive'],
        'positive_cell_capture_at_20pct': ranking['positive_cell_capture'],
        'burned_share_mass_capture_at_20pct': ranking['burned_share_mass_capture'],
    })
comparison = pd.DataFrame(rows).set_index('model')
display(comparison.style.format('{:.4f}'))

In [ ]:
year_rows = []
for model_name, result in metrics['metrics'].items():
    for year, values in result['by_final_test_year'].items():
        year_rows.append({'model': model_name, 'predictor_year': int(year), 'MAE': values['mae_all'], 'RMSE': values['rmse_all'], 'positive_row_MAE': values['mae_positive']})
by_year = pd.DataFrame(year_rows).sort_values(['predictor_year', 'model'])
display(by_year.style.format({'MAE': '{:.4f}', 'RMSE': '{:.4f}', 'positive_row_MAE': '{:.4f}'}))

## Annual operational use

For a forecast year `Y`, the model is refit only through labelled predictor year `Y−2`; an unlabelled feature matrix from `T=Y−1` is then scored. When ICNF later supplies the observed outcome for `Y`, that score can be evaluated and the unchanged specification refit for `Y+1`.

Use `python scripts/prepare_operational_forecast.py` and `python scripts/score_operational_forecast.py` only for the controlled annual cycle. The full project runner performs them in the correct order.